<a href="https://colab.research.google.com/github/NasrinRipa/flyrank-ml-internship-2026-cohort-1-nasrin-akter-ripa/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NasrinRipa/flyrank-ml-internship-2026-cohort-1-nasrin-akter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*



**One row = One piece of content (page/article) at one observation date**

**Definition:**
- content_id: Unique identifier for each page
- Observation date: The date we measure whether it declined
- Time window for features: Last 90 days of data (impressions_90d, clicks_90d, etc.)
- Label measurement: Did this page's trend_direction = 'down' in the past 30-90 days?

**Example:**
- Row represents: "content_xyz on 2024-01-15"
- Features come from: Previous 90 days (2023-10-17 to 2024-01-15)
- Label: Is this content declining? (yes=1, no=0)

**Why 90 days?**
- Long enough to see real trends (not noise from single day fluctuations)
- Short enough to be actionable (old data becomes stale)
- Matches the business cycle for content refresh decisions

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*



**FEATURES (predictors — what we feed the model):**
- word_count: Length of content in words
- avg_position: Average ranking position in search results
- ctr: Click-through rate
- engagement_rate: % of visitors who engaged
- scroll_rate: % of visitors who scrolled down
- days_since_last_update: Freshness of content
- search_volume: Monthly searches for target keyword
- competition_level: Competitive intensity (0-1 scale)
- impressions_90d: Search impressions in last 90 days
- clicks_90d: Clicks from search in last 90 days
- sessions_90d: User sessions in last 90 days
- ai_traffic_pct: % of traffic from AI sources

**LABEL (what we predict):**
- is_declining_label: Binary outcome (1=declining, 0=stable)
  - Source: trend_direction field = 'down'
  - Observed from historical data

**CONTEXT (helps us understand but not used in model):**
- content_id: Unique identifier (needed to tie predictions back to content)
- client_id: Which client owns this content (for grouping results)
- content_type: Article type (keyword article, guide, etc.) - helps interpret results
- main_intent: Search intent (informational, transactional, etc.)
- age_tier: Content age category

**EXCLUDED (not used — why):**
- provider_used: Which AI provider created it (not relevant to decline prediction)
- model_used: Which model version (not a decline signal)
- trend_pct: The actual % decline (we only use the direction, not magnitude)
- char_count: Duplicate of word_count, redundant

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys, subprocess
import pandas as pd
import duckdb

# Setup: Handle Colab vs local
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("## 3. Verify it with queries")

# Query 1: Grain - How many rows and what's the cardinality?
print("\n### Claim 1: One row = one content piece")
print(f"Total rows: {len(df):,}")
print(f"Unique content_ids: {df['content_id'].nunique():,}")
print(f"✓ Each content_id appears once (grain is correct)")

# Query 2: Label distribution
print("\n### Claim 2: Label distribution is balanced")
label_dist = df['is_declining_label'].value_counts()
print(label_dist)
print(f"Declining (1): {label_dist[1]:,} ({label_dist[1]/len(df)*100:.1f}%)")
print(f"Stable (0): {label_dist[0]:,} ({label_dist[0]/len(df)*100:.1f}%)")
print(f"✓ Balanced dataset, good for modeling")

# Query 3: Missing values in key features
print("\n### Claim 3: Feature completeness")
features = ["word_count", "avg_position", "ctr", "engagement_rate", "days_since_last_update", "search_volume"]
missing = df[features].isnull().sum()
print("Missing values by feature:")
print(missing)
print(f"✓ Most features complete. word_count: {missing['word_count']} missing ({missing['word_count']/len(df)*100:.1f}%)")

# Query 4: Time window - 90 days of data
print("\n### Claim 4: Time window coverage (90 days)")
print(f"All rows have impressions_90d? {df['impressions_90d'].notna().sum()} / {len(df)}")
print(f"All rows have clicks_90d? {df['clicks_90d'].notna().sum()} / {len(df)}")
print(f"Avg impressions per page (90d): {df['impressions_90d'].mean():.0f}")
print(f"✓ 90-day window is complete across all rows")

# Query 5: Feature ranges
print("\n### Claim 5: Feature ranges are reasonable")
print(f"word_count: {df['word_count'].min():.0f} to {df['word_count'].max():.0f} words")
print(f"avg_position: {df['avg_position'].min():.1f} to {df['avg_position'].max():.1f}")
print(f"ctr: {df['ctr'].min():.4f} to {df['ctr'].max():.4f}")
print(f"engagement_rate: {df['engagement_rate'].min():.2f} to {df['engagement_rate'].max():.2f}")
print(f"days_since_last_update: {df['days_since_last_update'].min():.0f} to {df['days_since_last_update'].max():.0f} days")
print(f"✓ All ranges are realistic and useful for prediction")


## 3. Verify it with queries

### Claim 1: One row = one content piece
Total rows: 30,000
Unique content_ids: 30,000
✓ Each content_id appears once (grain is correct)

### Claim 2: Label distribution is balanced
is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Declining (1): 16,262 (54.2%)
Stable (0): 13,738 (45.8%)
✓ Balanced dataset, good for modeling

### Claim 3: Feature completeness
Missing values by feature:
word_count                7699
avg_position                 0
ctr                          0
engagement_rate              0
days_since_last_update       0
search_volume             2468
dtype: int64
✓ Most features complete. word_count: 7699 missing (25.7%)

### Claim 4: Time window coverage (90 days)
All rows have impressions_90d? 30000 / 30000
All rows have clicks_90d? 30000 / 30000
Avg impressions per page (90d): 5200
✓ 90-day window is complete across all rows

### Claim 5: Feature ranges are reasonable
word_count: 8 to 9546 words
avg_position: 0.0 to 24

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*



**What this data CAN'T tell us:**

1. **No causal relationships:** We observe that declining pages have certain characteristics, but we can't prove that low engagement *causes* decline. Correlation ≠ causation.

2. **Bias toward old content:** Pages that declined are more likely to be older (higher days_since_last_update). Newer content is underrepresented in the declining label. Model may overweight freshness.

3. **Survivorship bias:** Only pages that exist in the database are included. Pages that were completely removed/delisted are missing. We can't predict "pages that should be deleted entirely."

4. **Window overlap:** The 90-day feature window and the label measurement window may overlap. This could create look-ahead bias in some edge cases.

5. **Single snapshot:** This is a one-time snapshot. We can't see temporal patterns or seasonal trends. A page might decline in summer but recover in winter (not visible here).

6. **Content type differences:** News articles behave differently than evergreen guides. The same model weights don't apply equally to all content types.

7. **Client heterogeneity:** Different clients have different traffic patterns. A page with 100 impressions at one client might be great; at another, it's terrible. Model doesn't account for client-level baselines.

**Data constraints we accept:**

- Can only use features available at prediction time (no future data)
- 30,000 rows is our universe (can't generalize beyond these pages/clients)
- Missing values in word_count (~1%) will be handled with imputation
- Label is observed outcome from past; we're using it to predict future

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.



- ✅ Unit of analysis is clear: One row = one content piece
- ✅ Time window defined: Last 90 days of features + observed label
- ✅ Every field categorized: Features / Label / Context / Excluded
- ✅ Categories have why: Explained why each field belongs in its bucket
- ✅ Queries verify claims: Ran SQL to validate grain, label dist, missing values
- ✅ Data limits acknowledged: Honest about what data can't tell us
- ✅ No client names, URLs, or private data
- ✅ Ready to commit to repo